# Week 7 – Delta Lake 

## Objective
This notebook demonstrates an end-to-end incremental data processing pipeline using Delta Lake in Azure Databricks. The workflow includes loading customer data from CSV files, performing data quality checks, cleaning and transforming records, creating a Delta table, processing incremental updates, and validating the final dataset after applying a Delta Lake MERGE operation.

---

## Dataset Information

### Master Dataset
Contains the initial customer records and includes:
- Missing values
- Duplicate customer records

### Incremental Dataset
Contains:
- Updated customer information
- New customer records
- Duplicate records
- Missing values

---

## Workflow

1. Load and explore the master dataset.
2. Identify missing values and duplicate records.
3. Clean and standardize customer data.
4. Store cleaned data as a Delta table.
5. Load and clean incremental customer data.
6. Apply Delta Lake MERGE (SCD Type 1).
7. Validate updates and inserts.
8. Verify final record count and data quality.

---

## Expected Outcome

- Missing values replaced with appropriate defaults.
- Duplicate customer IDs removed.
- Existing customer records updated.
- New customer records inserted.
- Final Delta table contains 13 unique customer records.
- No duplicate customer IDs remain after processing.

---

### Submitted by
#### **Yadnyesh Sawant**  
- **StudentId: CT_CSI_DE_1015**
- MCA, MIT World Peace University  
- CEI Data Engineering Internship – Week 7 Assignment


In [0]:
from pyspark.sql import functions as sqlF
from delta.tables import DeltaTable
print("Libraries imported successfully")

Libraries imported successfully


In [0]:
master_csv_path = "/Volumes/delta_scd_assignment/default/week7_data/customer_master.csv"
incremental_csv_path = "/Volumes/delta_scd_assignment/default/week7_data/customer_incremental.csv"
delta_table_path = "/Volumes/delta_scd_assignment/default/week7_data/customer_delta"

## Step 1: Load Master Dataset


In [0]:
master_df = spark.read.csv(
    master_csv_path,
    header=True,
    inferSchema=True
)

print(f"Total Records : {master_df.count()}")

display(master_df)

Total Records : 11


customer_id,customer_name,email,city,phone,updated_at
101,Aarav Sharma,aarav@gmail.com,Mumbai,9876500001,2026-07-01
102,Priya Patel,priya@gmail.com,Ahmedabad,9876500002,2026-07-02
103,Rohan Gupta,rohan@gmail.com,Delhi,9876500003,2026-07-03
104,Sneha Reddy,sneha@gmail.com,Hyderabad,9876500004,2026-07-04
105,Vikram Singh,vikram@gmail.com,Jaipur,9876500005,2026-07-05
106,Ananya Iyer,ananya@gmail.com,Chennai,9876500006,2026-07-06
107,Karan Mehta,karan@gmail.com,Pune,9876500007,2026-07-07
108,Divya Nair,divya@gmail.com,null,9876500008,2026-07-08
109,Arjun Rao,arjun@gmail.com,Bengaluru,9876500009,2026-07-09
110,Neha Joshi,neha@gmail.com,Nagpur,9876500010,2026-07-10


In [0]:
master_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- phone: long (nullable = true)
 |-- updated_at: date (nullable = true)



## Step 2: Data Quality Assessment

In [0]:
master_df.select([
    F.count(
        F.when(
            F.col(column).isNull() |
            (F.trim(F.col(column).cast("String")) == ""),
            column
        )y
    ).alias(column)
    for column in master_df.columns
]).show()

+-----------+-------------+-----+----+-----+----------+
|customer_id|customer_name|email|city|phone|updated_at|
+-----------+-------------+-----+----+-----+----------+
|          0|            0|    0|   1|    0|         0|
+-----------+-------------+-----+----+-----+----------+



In [0]:
duplicate_rows = (
    master_df.groupBy("customer_id")
             .count()
             .filter("count > 1")
)

display(duplicate_rows)

customer_id,count
110,2


## Step 3: Data Cleaning

In [0]:
master_clean = (
    master_df
        .fillna({"city":"Unknown"})
        .dropDuplicates(["customer_id"])
)

master_clean = (
    master_clean
        .withColumn(
            "customer_name",
            sqlF.trim("customer_name")
        )
        .withColumn(
            "email",
            sqlF.lower(sqlF.trim("email"))
        )
)

display(master_clean.orderBy("customer_id"))

customer_id,customer_name,email,city,phone,updated_at
101,Aarav Sharma,aarav@gmail.com,Mumbai,9876500001,2026-07-01
102,Priya Patel,priya@gmail.com,Ahmedabad,9876500002,2026-07-02
103,Rohan Gupta,rohan@gmail.com,Delhi,9876500003,2026-07-03
104,Sneha Reddy,sneha@gmail.com,Hyderabad,9876500004,2026-07-04
105,Vikram Singh,vikram@gmail.com,Jaipur,9876500005,2026-07-05
106,Ananya Iyer,ananya@gmail.com,Chennai,9876500006,2026-07-06
107,Karan Mehta,karan@gmail.com,Pune,9876500007,2026-07-07
108,Divya Nair,divya@gmail.com,Unknown,9876500008,2026-07-08
109,Arjun Rao,arjun@gmail.com,Bengaluru,9876500009,2026-07-09
110,Neha Joshi,neha@gmail.com,Nagpur,9876500010,2026-07-10


## Step 4: Create Delta Table

In [0]:
master_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_table_path)

print("Delta Table Created")

Delta Table Created


In [0]:
delta_customers = spark.read.format("delta").load(delta_table_path)

print(delta_customers.count())

display(delta_customers)

10


customer_id,customer_name,email,city,phone,updated_at
105,Vikram Singh,vikram@gmail.com,Jaipur,9876500005,2026-07-05
104,Sneha Reddy,sneha@gmail.com,Hyderabad,9876500004,2026-07-04
108,Divya Nair,divya@gmail.com,Unknown,9876500008,2026-07-08
109,Arjun Rao,arjun@gmail.com,Bengaluru,9876500009,2026-07-09
106,Ananya Iyer,ananya@gmail.com,Chennai,9876500006,2026-07-06
103,Rohan Gupta,rohan@gmail.com,Delhi,9876500003,2026-07-03
107,Karan Mehta,karan@gmail.com,Pune,9876500007,2026-07-07
102,Priya Patel,priya@gmail.com,Ahmedabad,9876500002,2026-07-02
101,Aarav Sharma,aarav@gmail.com,Mumbai,9876500001,2026-07-01
110,Neha Joshi,neha@gmail.com,Nagpur,9876500010,2026-07-10


## Step 5: Load Incremental Dataset

In [0]:
inc_df = spark.read.csv(
    incremental_csv_path,
    header=True,
    inferSchema=True
)

display(inc_df)

customer_id,customer_name,email,city,phone,updated_at
102,Priya Patel,priya.patel@gmail.com,Pune,9876500002,2026-08-01
104,Sneha Reddy,sneha.reddy@gmail.com,Bengaluru,9876500004,2026-08-01
107,Karan Mehta,karan@gmail.com,Mumbai,9876500007,2026-08-01
111,Riya Kapoor,riya@gmail.com,Pune,9876500011,2026-08-01
112,Aditya Kulkarni,aditya@gmail.com,null,9876500012,2026-08-01
113,Meera Shah,meera@gmail.com,Surat,9876500013,2026-08-01
113,Meera Shah,meera@gmail.com,Surat,9876500013,2026-08-01


## Step 6: Clean Incremental Dataset

In [0]:
inc_clean = (
    inc_df
        .fillna({"city":"Unknown"})
        .dropDuplicates(["customer_id"])
)

inc_clean = (
    inc_clean
        .withColumn(
            "customer_name",
            sqlF.trim("customer_name")
        )
        .withColumn(
            "email",
            sqlF.lower(sqlF.col("email"))
        )
)

display(inc_clean.orderBy("customer_id"))

customer_id,customer_name,email,city,phone,updated_at
102,Priya Patel,priya.patel@gmail.com,Pune,9876500002,2026-08-01
104,Sneha Reddy,sneha.reddy@gmail.com,Bengaluru,9876500004,2026-08-01
107,Karan Mehta,karan@gmail.com,Mumbai,9876500007,2026-08-01
111,Riya Kapoor,riya@gmail.com,Pune,9876500011,2026-08-01
112,Aditya Kulkarni,aditya@gmail.com,Unknown,9876500012,2026-08-01
113,Meera Shah,meera@gmail.com,Surat,9876500013,2026-08-01


## Step 7: Source Data Before MERGE

In [0]:
display(
    inc_clean.orderBy("customer_id")
)

customer_id,customer_name,email,city,phone,updated_at
102,Priya Patel,priya.patel@gmail.com,Pune,9876500002,2026-08-01
104,Sneha Reddy,sneha.reddy@gmail.com,Bengaluru,9876500004,2026-08-01
107,Karan Mehta,karan@gmail.com,Mumbai,9876500007,2026-08-01
111,Riya Kapoor,riya@gmail.com,Pune,9876500011,2026-08-01
112,Aditya Kulkarni,aditya@gmail.com,Unknown,9876500012,2026-08-01
113,Meera Shah,meera@gmail.com,Surat,9876500013,2026-08-01


## Step 8: Delta Lake MERGE Operation

In [0]:
customer_delta = DeltaTable.forPath(
    spark,
    delta_table_path
)

(
    customer_delta.alias("t")
    .merge(
        inc_clean.alias("s"),
        "t.customer_id = s.customer_id"
    )
    .whenMatchedUpdate(
        set={y
            "customer_name":"s.customer_name",
            "email":"s.email",
            "city":"s.city",
            "phone":"s.phone",
            "updated_at":"s.updated_at"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("UPSERT Successful")

UPSERT Successful


## Step 9: Validation

In [0]:
final_df = spark.read \
    .format("delta") \
    .load(delta_table_path)

print("Final Count :", final_df.count())

display(
    final_df.orderBy("customer_id")
)

Final Count : 13


customer_id,customer_name,email,city,phone,updated_at
101,Aarav Sharma,aarav@gmail.com,Mumbai,9876500001,2026-07-01
102,Priya Patel,priya.patel@gmail.com,Pune,9876500002,2026-08-01
103,Rohan Gupta,rohan@gmail.com,Delhi,9876500003,2026-07-03
104,Sneha Reddy,sneha.reddy@gmail.com,Bengaluru,9876500004,2026-08-01
105,Vikram Singh,vikram@gmail.com,Jaipur,9876500005,2026-07-05
106,Ananya Iyer,ananya@gmail.com,Chennai,9876500006,2026-07-06
107,Karan Mehta,karan@gmail.com,Mumbai,9876500007,2026-08-01
108,Divya Nair,divya@gmail.com,Unknown,9876500008,2026-07-08
109,Arjun Rao,arjun@gmail.com,Bengaluru,9876500009,2026-07-09
110,Neha Joshi,neha@gmail.com,Nagpur,9876500010,2026-07-10


In [0]:
duplicate_check = (
    final_df
        .groupBy("customer_id")
        .count()
        .filter("count > 1")
)

print(
    "Duplicate Records :",
    duplicate_check.count()
)

Duplicate Records : 0


In [0]:
display(
    final_df.filter(
        sqlF.col("customer_id").isin(
            102,104,107
        )
    )
)

customer_id,customer_name,email,city,phone,updated_at
104,Sneha Reddy,sneha.reddy@gmail.com,Bengaluru,9876500004,2026-08-01
107,Karan Mehta,karan@gmail.com,Mumbai,9876500007,2026-08-01
102,Priya Patel,priya.patel@gmail.com,Pune,9876500002,2026-08-01


In [0]:
display(
    final_df.filter(
        sqlF.col("customer_id").isin(
            111,112,113
        )
    )
)

customer_id,customer_name,email,city,phone,updated_at
111,Riya Kapoor,riya@gmail.com,Pune,9876500011,2026-08-01
112,Aditya Kulkarni,aditya@gmail.com,Unknown,9876500012,2026-08-01
113,Meera Shah,meera@gmail.com,Surat,9876500013,2026-08-01


In [0]:
summary = [
    "Master data loaded",
    "Missing values handled",
    "Duplicates removed",
    "Delta table created",
    "Incremental data loaded",
    "MERGE operation completed",
    "Updates applied",
    "New customers inserted",
    "Validation successful"
]

for item in summary:
    print("✓", item)

✓ Master data loaded
✓ Missing values handled
✓ Duplicates removed
✓ Delta table created
✓ Incremental data loaded
✓ MERGE operation completed
✓ Updates applied
✓ New customers inserted
✓ Validation successful


In [0]:
y